# 누적 점수 부여와 전체 연도 통합

원본 최종 수정 기록: 2025-12-09T21:10:28+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd
import os

# 설정
YEARS = [2021, 2022, 2023, 2024, 2025]
BASE_PATH = './data/kaggle-dataset'

def add_cumulative_score_for_year(year):
    """
    특정 년도의 모든 게임에 누적 점수(Score) 추가
    
    Step 1: Score 계산
    - Win → +1점
    - Loss → +0점
    - 누적해서 라운드별 스코어 계산
    """
    
    print(f"\n{'='*80}")
    print(f"VCT {year} - Step 1: Score 계산 (라운드별 누적 점수)")
    print(f"{'='*80}")
    
    year_path = f"{BASE_PATH}/vct_{year}"
    matches_path = f"{year_path}/matches"
    
    input_file = f"{matches_path}/rounds_combined_clean.csv"
    output_file = f"{matches_path}/rounds_with_score.csv"
    
    # 1. 파일 확인
    print(f"\n[1/5] 파일 확인 중...")
    if not os.path.exists(input_file):
        print(f"❌ 입력 파일 없음: {input_file}")
        return False
    
    print(f"✅ 입력 파일 존재: {input_file}")
    
    # 2. 데이터 로드
    print(f"\n[2/5] 데이터 로드 중...")
    try:
        df = pd.read_csv(input_file)
        print(f"✅ 로드 완료: {len(df):,}개 행, {df['Game ID'].nunique()}개 게임")
    except Exception as e:
        print(f"❌ 로드 오류: {str(e)}")
        return False
    
    # 3. Outcome 검증
    print(f"\n[3/5] Outcome 데이터 검증...")
    outcome_values = df['Outcome'].unique()
    print(f"   Outcome 값: {outcome_values}")
    
    win_count = (df['Outcome'] == 'Win').sum()
    loss_count = (df['Outcome'] == 'Loss').sum()
    
    print(f"   Win 개수: {win_count:,}")
    print(f"   Loss 개수: {loss_count:,}")
    
    if win_count == 0 or loss_count == 0:
        print(f"⚠️  경고: Win 또는 Loss 데이터가 부족합니다.")
    
    # 4. 누적 점수 계산
    print(f"\n[4/5] 라운드별 누적 점수 계산 중...")
    
    result_list = []
    game_count = 0
    error_count = 0
    
    for game_id, game_data in df.groupby('Game ID'):
        try:
            # 게임별로 팀별로 처리
            for team in game_data['Team'].unique():
                team_data = game_data[game_data['Team'] == team].sort_values('Round Number').reset_index(drop=True)
                
                if len(team_data) == 0:
                    continue
                
                # 누적 점수 계산
                team_data = team_data.copy()
                team_data['Cumulative Score'] = (team_data['Outcome'] == 'Win').astype(int).cumsum()
                
                # 추가 검증 정보
                team_data['Win Count'] = (team_data['Outcome'] == 'Win').cumsum()
                team_data['Loss Count'] = (team_data['Outcome'] == 'Loss').cumsum()
                
                result_list.append(team_data)
            
            game_count += 1
            
        except Exception as e:
            print(f"   ⚠️  게임 {game_id} 처리 중 오류: {str(e)}")
            error_count += 1
            continue
    
    result_df = pd.concat(result_list, ignore_index=True)
    result_df = result_df.sort_values(['Game ID', 'Round Number', 'Team']).reset_index(drop=True)
    
    print(f"✅ 계산 완료: {game_count}개 게임, 오류: {error_count}개")
    
    # 5. 데이터 검증 및 저장
    print(f"\n[5/5] 데이터 검증 및 저장...")
    
    print(f"\n검증:")
    print(f"   총 행 수: {len(result_df):,}")
    print(f"   게임 수: {result_df['Game ID'].nunique()}")
    print(f"   라운드 범위: {result_df['Round Number'].min()} ~ {result_df['Round Number'].max()}")
    print(f"   최종 스코어 범위: {result_df['Cumulative Score'].min()} ~ {result_df['Cumulative Score'].max()}")
    
    # 샘플 게임 검증
    sample_game = result_df['Game ID'].iloc[0]
    sample_data = result_df[result_df['Game ID'] == sample_game]
    
    print(f"\n샘플 게임 ({sample_game}):")
    for team in sample_data['Team'].unique():
        team_rounds = sample_data[sample_data['Team'] == team].sort_values('Round Number')
        final_score = team_rounds['Cumulative Score'].iloc[-1]
        print(f"  {team}: 최종 스코어 = {final_score}, 라운드 수 = {len(team_rounds)}")
    
    # 파일 저장
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ 저장 완료: {output_file}")
    
    return True

def main():
    print("="*80)
    print("VCT 2021-2025 Step 1: 라운드별 누적 점수(Score) 계산")
    print("="*80)
    
    success_count = 0
    fail_count = 0
    
    for year in YEARS:
        try:
            if add_cumulative_score_for_year(year):
                success_count += 1
            else:
                fail_count += 1
        except Exception as e:
            print(f"\n❌ VCT {year} 처리 중 오류: {str(e)}")
            fail_count += 1
    
    # 최종 요약
    print(f"\n{'='*80}")
    print("최종 결과")
    print(f"{'='*80}")
    print(f"✅ 성공: {success_count}개 년도")
    print(f"❌ 실패: {fail_count}개 년도")
    
    if success_count == len(YEARS):
        print(f"\n🎉 모든 년도 처리 완료!")
        print(f"\n다음 단계:")
        print(f"  Step 2: Seq2Seq 입력 데이터 구성")
        print(f"  - Round Index")
        print(f"  - My Score (내 누적 점수)")
        print(f"  - Opp Score (상대 누적 점수)")
        print(f"  - My Bank (자금)")
        print(f"  - Loadout Value (장비가치)")
        print(f"  - Buy Type (구매유형)")
        print(f"  - My Kills / Opp Kills")
        print(f"  - Round Outcome")
    
    return success_count == len(YEARS)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os
from pathlib import Path

# 설정
YEARS = [2021, 2022, 2023, 2024, 2025]
BASE_PATH = './data/kaggle-dataset'
OUTPUT_PATH = './data/kaggle-dataset/vct_combined_all_years.csv'

def merge_all_years():
    """
    VCT 2021-2025 모든 년도의 데이터를 하나의 CSV로 통합
    
    각 년도의 rounds_with_score.csv를 읽어서 하나로 합치기
    """
    
    print("="*80)
    print("VCT 2021-2025 데이터 통합")
    print("="*80)
    
    # 1. 각 년도별 파일 경로 확인
    print("\n[1/4] 파일 경로 확인 중...")
    
    files_to_merge = []
    missing_years = []
    
    for year in YEARS:
        year_path = f"{BASE_PATH}/vct_{year}"
        matches_path = f"{year_path}/matches"
        file_path = f"{matches_path}/rounds_with_score.csv"
        
        if os.path.exists(file_path):
            file_size_mb = os.path.getsize(file_path) / (1024*1024)
            print(f"  ✅ VCT {year}: {file_size_mb:.1f}MB - {file_path}")
            files_to_merge.append(file_path)
        else:
            print(f"  ❌ VCT {year}: 파일 없음 - {file_path}")
            missing_years.append(year)
    
    if len(missing_years) > 0:
        print(f"\n⚠️  경고: {missing_years} 년도 파일이 없습니다.")
        print(f"   먼저 step1_add_score_all_years.py를 실행하세요.")
        return False
    
    # 2. 각 파일 로드 및 통합
    print(f"\n[2/4] 데이터 로드 및 통합 중...")
    
    all_dfs = []
    total_rows = 0
    total_games = 0
    
    for idx, file_path in enumerate(files_to_merge, 1):
        try:
            print(f"\n  [{idx}/{len(files_to_merge)}] 로드 중: {file_path}")
            df = pd.read_csv(file_path)
            
            # 년도 정보 추가
            year = YEARS[idx-1]
            df['Year'] = year
            
            rows = len(df)
            games = df['Game ID'].nunique()
            
            print(f"      ✅ 로드 완료: {rows:,}개 행, {games}개 게임")
            
            all_dfs.append(df)
            total_rows += rows
            total_games += games
            
        except Exception as e:
            print(f"      ❌ 오류: {str(e)}")
            return False
    
    # 3. 데이터 통합
    print(f"\n[3/4] 데이터 통합 중...")
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(['Year', 'Game ID', 'Round Number', 'Team']).reset_index(drop=True)
    
    print(f"✅ 통합 완료:")
    print(f"   총 행 수: {len(combined_df):,}개")
    print(f"   총 게임 수: {combined_df['Game ID'].nunique():,}개")
    print(f"   년도별 게임 수:")
    
    for year in YEARS:
        year_games = combined_df[combined_df['Year'] == year]['Game ID'].nunique()
        year_rows = len(combined_df[combined_df['Year'] == year])
        print(f"      VCT {year}: {year_games}개 게임, {year_rows:,}개 행")
    
    # 4. 데이터 검증
    print(f"\n[4/4] 데이터 검증...")
    
    print(f"\n컬럼 확인:")
    print(f"   {combined_df.columns.tolist()}")
    
    print(f"\n데이터 통계:")
    print(f"   Cumulative Score 범위: {combined_df['Cumulative Score'].min()} ~ {combined_df['Cumulative Score'].max()}")
    print(f"   라운드 범위: {combined_df['Round Number'].min()} ~ {combined_df['Round Number'].max()}")
    print(f"   My_Kills 범위: {combined_df['My_Kills'].min()} ~ {combined_df['My_Kills'].max()}")
    print(f"   Opp_Kills 범위: {combined_df['Opp_Kills'].min()} ~ {combined_df['Opp_Kills'].max()}")
    
    print(f"\nOutcome 분포:")
    outcome_dist = combined_df['Outcome'].value_counts()
    for outcome, count in outcome_dist.items():
        percentage = (count / len(combined_df)) * 100
        print(f"   {outcome}: {count:,}개 ({percentage:.1f}%)")
    
    # 샘플 데이터 확인
    print(f"\n샘플 데이터 (처음 10행):")
    sample_cols = ['Year', 'Game ID', 'Round Number', 'Team', 'Opponent_Team', 
                   'Cumulative Score', 'My_Kills', 'Opp_Kills', 'Outcome']
    print(combined_df[sample_cols].head(10).to_string(index=False))
    
    # 5. 저장
    print(f"\n저장 중...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    combined_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f"✅ 저장 완료: {OUTPUT_PATH}")
    
    # 최종 요약
    print(f"\n{'='*80}")
    print("✅ 통합 완료!")
    print(f"{'='*80}")
    print(f"\n통합 파일:")
    print(f"  {OUTPUT_PATH}")
    print(f"\n통계:")
    print(f"  총 행: {len(combined_df):,}")
    print(f"  총 게임: {combined_df['Game ID'].nunique():,}")
    print(f"  파일 크기: {os.path.getsize(OUTPUT_PATH) / (1024*1024):.1f}MB")
    
    print(f"\n다음 단계:")
    print(f"  Step 2: Seq2Seq 입력 데이터 구성")
    print(f"  - 전반 12라운드 시퀀스 추출")
    print(f"  - 후반 12라운드 타겟 생성")
    print(f"  - 정규화 및 전처리")
    
    return True

if __name__ == "__main__":
    merge_all_years()